# Tabular 01 · Data cleaning, specialist-reviewed grades and missing findings

**Objective:** turn the eye-bank research dataset into the 24 modelling features and the target grade (A+, A, B), keeping a missing clinical finding distinct from a negative one.

**Derived from:** `notebooks/01_data_loading.ipynb`, `notebooks/02_data_cleaning.ipynb` (only the steps that affect the target and the 24 features; recipient, graft-size and request fields cleaned in 02 are not used for grading and are omitted).
**Output:** nothing is written. The cleaning is rebuilt in memory and checked cell-for-cell against the saved `data/processed/cleaned_grading_dataset.xlsx` that every later notebook uses. **Runtime:** under 1 minute.

> Notebook 01's saved output shows an older version of the raw file (with a "Labelling part 2" section and rule-predicted grades). The current raw file contains only eye-bank-form grades, as notebook 02 and this notebook show.

Set up paths and load the raw research dataset.

In [1]:
import re
import sys
from pathlib import Path

GRADING_DIR = next(p for p in [Path.cwd(), *Path.cwd().parents] if (p / "scripts" / "modeling_setup.py").is_file())
sys.path[:0] = [str(GRADING_DIR / "scripts"), str(GRADING_DIR / "notebooks_cleaned")]

import numpy as np
import pandas as pd

import cleaned_setup as cs
import modeling_setup as ms

RAW_PATH = GRADING_DIR / "data" / "combined_dataset_labeled.xlsx"
df = pd.read_excel(RAW_PATH, sheet_name="Research Dataset")
print(f"{RAW_PATH.name}: sha256 {cs.sha256(RAW_PATH)}… | shape {df.shape}")

combined_dataset_labeled.xlsx: sha256 0a199ac418381bbe… | shape (1055, 84)


## 1. Target grade

Every recorded grade comes from an eye-bank form. Most are a plain A+, A or B; some carry annotations from the source forms (conflicting release form and tissue sheet, handwritten changes, "verify").

In [2]:
standard = df["Tissue_Grade"].isin(cs.CLASSES)
print("Grade source:", df["Tissue_Grade_Source"].value_counts(dropna=False).to_dict())
print(f"Standard grades: {standard.sum()} | annotated: {(~standard & df['Tissue_Grade'].notna()).sum()} | missing: {df['Tissue_Grade'].isna().sum()}")

Grade source: {'Eye-bank form': 1037, nan: 18}
Standard grades: 1021 | annotated: 15 | missing: 19


The 15 annotated grades were resolved by specialist review (mapping from notebook 02, keyed by tissue serial). The original text is kept; the reviewed grade goes into `Tissue_Grade_Clean`.

In [3]:
review_grades = {
    "12729": "A", "16056": "A", "16057": "A", "16060": "A", "16061": "B",
    "16074": "A+", "16133": "B", "16252": "A+", "16254": "B", "16297": "A",
    "16603": "A", "17466": "A", "17482": "B", "17495": "B", "17520": "A",
}
df["Tissue_Grade_Clean"] = df["Tissue_Grade"]
serial = df["Tissue_ID"].astype(str).str.extract(r"(\d{5})")[0]
reviewed = serial.isin(review_grades)
df.loc[reviewed, "Tissue_Grade_Clean"] = serial[reviewed].map(review_grades)
assert (~standard & df["Tissue_Grade"].notna()).sum() == reviewed.sum() == 15
df.loc[reviewed, ["Tissue_Grade", "Tissue_Grade_Clean"]].rename(columns={"Tissue_Grade": "Recorded annotation", "Tissue_Grade_Clean": "Specialist-reviewed grade"}).sort_values("Specialist-reviewed grade")

,Recorded annotation,Specialist-reviewed grade
74,A (as read - verify),A
226,A (release form; tissue sheet A+),A
227,A (release form; tissue sheet A+),A
230,A+ / A (both circled on tissue sheet - verify),A
364,A (release form; tissue sheet A+),A
371,A+ / A (both circled),A
414,A+ (as read),A
468,A (release form; tissue sheet circles A+),A
244,A (tissue sheet) / A+ (release form),A+
319,A (tissue sheet) / A+ (release form),A+


Grade counts after review. Tissues without any recorded grade stay missing and are excluded from modelling (never imputed).

In [4]:
df["Tissue_Grade_Clean"].value_counts(dropna=False).rename("Tissues").to_frame()

,Tissues
Tissue_Grade_Clean,
A+,545
A,319
B,172
NaN,19


## 2. Konan endothelial morphology (6 measurements)

Four records have every Konan value equal to 0, which is physically impossible (no cells counted): they are set to missing, not kept as zeros.

In [5]:
invalid = df["ECD_CD_cells_per_mm2"] == 0
print("All-zero Konan records:", int(invalid.sum()), "| all six values zero:", bool((df.loc[invalid, ms.NUMERIC_FEATURES] == 0).all().all()))
df.loc[invalid, ms.NUMERIC_FEATURES] = np.nan

All-zero Konan records: 4 | all six values zero: True


One tissue has every Konan value except SD. Notebook 02 entered SD = 229 for it (the notebook does not record the source of this value).

In [6]:
sd_gap = df["ECD_CD_cells_per_mm2"].notna() & df["SD"].isna()
print("Tissues with ECD but no SD:", int(sd_gap.sum()))
df.loc[sd_gap, "SD"] = 229

Tissues with ECD but no SD: 1


## 3. Slit-lamp findings (18 features) from free text

Each finding is parsed from its layer's free-text column. **If the column is empty, the finding is missing (`NA`), not "No":** an unrecorded examination is different from a recorded normal one. `False` is used only when text was recorded and the finding is absent or negated.

In [7]:
def flag(pattern, negation=None):
    """True/False from recorded text; NA when nothing was recorded."""
    def parse(value):
        if pd.isna(value):
            return pd.NA
        text = str(value).lower()
        if negation and re.search(negation, text):
            return False
        return bool(re.search(pattern, text))
    return parse


STROMA, EPI, DESCEMET, ENDO = "Stroma_Findings", "Epithelium_Findings", "Descemet_Findings", "Endothelium_Findings"
BINARY_PARSERS = {
    "Stroma_Is_Clear": (STROMA, flag(r"\bclear\b")),
    "Stroma_Is_Compact": (STROMA, flag(r"\bcompact\b")),
    "Stroma_Has_Arcus": (STROMA, flag(r"\barcus\b")),
    "Stroma_Has_Edema": (STROMA, flag(r"\bedema\b", negation=r"\bno\s+edema\b")),
    "Stroma_Has_Scar": (STROMA, flag(r"\bscar\b")),
    "Epi_Has_Arcus": (EPI, flag(r"\barcus\b")),
    "Epi_Has_Defect": (EPI, flag(r"\b(epithelial\s+)?defect\b", negation=r"\bno\s+(epithelial\s+)?defect\b")),
    "Epi_Has_Exposure": (EPI, flag(r"exposure", negation=r"\bno exposure\b")),
    "Epi_Has_Haze": (EPI, flag(r"\bhaze\b", negation=r"\bno\s+haze\b")),
    "Endo_Has_Stress_Lines": (ENDO, flag(r"stress lines", negation=r"no stress lines")),
    "Endo_Has_RBC": (ENDO, flag(r"\brbc")),
    "Endo_Has_Cell_Dropout": (ENDO, flag(r"cell\s*drop[- ]?out")),
}
for feature, (source, parse) in BINARY_PARSERS.items():
    df[feature] = df[source].apply(parse)

Descemet folds: entries with ambiguous marks ("verify", "both ticked", "struck", "no folds" alongside "folds"…) are flagged for review, and their fold features are set to missing rather than guessed.

In [8]:
REVIEW_TERMS = ["verify", "ambiguous", "both marked", "both ticked", "struck", "not ticked", "box not ticked", "box unticked"]


def descemet_review(value):
    if pd.isna(value):
        return pd.NA
    text = str(value).lower()
    return any(term in text for term in REVIEW_TERMS) or ("no folds" in text and "folds" in text)


def has_folds(value):
    if pd.isna(value):
        return pd.NA
    text = str(value).lower()
    if "no folds" in text and "folds" not in text.replace("no folds", ""):
        return False
    return "folds" in text


def fold_severity(value):
    if pd.isna(value):
        return pd.NA
    text = str(value).lower()
    if "verify" in text or "ambiguous" in text:
        return pd.NA
    for word, label in [("moderate", "Moderate"), ("very mild", "Very mild"), ("mild", "Mild")]:
        if word in text:
            return label
    return "Unspecified"


def fold_relative_number(value):
    if pd.isna(value):
        return pd.NA
    text = str(value).lower()
    if "relative number" not in text:
        return "Unspecified"
    for word, label in [("several/moderate", "Several/Moderate"), ("several", "Several"), ("moderate", "Moderate"), ("mild", "Mild")]:
        if word in text:
            return label
    return "Unspecified"


review = df[DESCEMET].apply(descemet_review)
df["Descemet_Has_Folds"] = df[DESCEMET].apply(has_folds)
df["Descemet_Fold_Severity"] = df[DESCEMET].apply(fold_severity)
df["Descemet_Fold_Relative_Number"] = df[DESCEMET].apply(fold_relative_number)
df.loc[review == True, ["Descemet_Has_Folds", "Descemet_Fold_Severity", "Descemet_Fold_Relative_Number"]] = pd.NA
print("Descemet entries flagged for review (set to missing):", int((review == True).sum()))

Descemet entries flagged for review (set to missing): 42


The remaining categorical findings: arcus severity, stress-line location and exposure severity ("None" when text was recorded without that finding).

In [9]:
def arcus_severity(value):
    if pd.isna(value):
        return pd.NA
    text = str(value).lower()
    if "arcus" not in text:
        return "None"
    return "Moderate" if "moderate arcus" in text else "Mild" if "mild arcus" in text else "Unspecified"


def stress_line_location(value):
    if pd.isna(value):
        return pd.NA
    text = str(value).lower()
    if "stress lines" not in text:
        return "None"
    if "central/peripheral" in text or "central and peripheral" in text:
        return "Central + Peripheral"
    return "Central" if "central" in text else "Peripheral" if "peripheral" in text else "Unspecified"


def exposure_severity(value):
    if pd.isna(value):
        return pd.NA
    text = str(value).lower()
    if "no exposure" in text:
        return "None"
    if "very mild" in text and "exposure" in text:
        return "Very mild"
    if "mild exposure" in text:
        return "Mild"
    return "Unspecified" if "exposure" in text else "None"


df["Stroma_Arcus_Severity"] = df[STROMA].apply(arcus_severity)
df["Endo_Stress_Line_Location"] = df[ENDO].apply(stress_line_location)
df["Epi_Exposure_Severity"] = df[EPI].apply(exposure_severity)

Three states per binary finding: recorded present, recorded absent, not recorded.

In [10]:
pd.DataFrame({f: df[f].map({True: "Present", False: "Absent"}).fillna("Not recorded").value_counts()
              for f in ["Stroma_Has_Scar", "Endo_Has_Cell_Dropout", "Descemet_Has_Folds", "Epi_Has_Haze"]}).fillna(0).astype(int)

,Stroma_Has_Scar,Endo_Has_Cell_Dropout,Descemet_Has_Folds,Epi_Has_Haze
Absent,555,482,9,555
Not recorded,467,480,511,468
Present,33,93,535,32


## 4. Columns that must never become features

Post-decision and downstream fields (suitability, outcome, requests, surgery) and `Additional_Findings`, which contains release-form grades, are excluded. Confirm none is among the 24 features.

In [11]:
LEAKAGE_COLUMNS = [
    "Overall_Donor_Quality", "Transplant_Suitability", "Suitability_Reason", "Tissue_Outcome", "Recipient_Age",
    "Recipient_Eye", "Recipient_Indication", "Request_Indication", "Primary_Procedure", "Secondary_Procedures",
    "Surgery_Date", "Requested_Grade", "Planned_Procedure", "Grade_Request_Met", "Review_Notes", "Additional_Findings",
]
grade_text = df["Additional_Findings"].dropna().str.contains(r"\bGrade\s*(?:A\+|A|B)\b", case=False, regex=True)
print(f"Additional_Findings entries quoting a grade: {grade_text.sum()} of {len(grade_text)}")
print("Excluded columns among the features:", sorted(set(LEAKAGE_COLUMNS) & set(ms.ALL_FEATURES)) or "none")

Additional_Findings entries quoting a grade: 255 of 587
Excluded columns among the features: none


Notebook 02 removed one incomplete record (no grade, unknown year/month).

In [12]:
df = df[df["Tissue_ID"] != "14302 L (year/month unknown)"].reset_index(drop=True)
print("Records:", len(df))

Records: 1054


## 5. Reproduction check against the saved cleaned dataset

Every value of the target and the 24 features must equal the saved file (missing values must be missing in both). The file is read literally here (only empty cells count as missing; see Section 6 for why that matters).

In [13]:
saved = pd.read_excel(ms.DATA_PATH, keep_default_na=False, na_values=[""])
assert list(saved["Tissue_ID"]) == list(df["Tissue_ID"]), "Record order differs"


def as_float(series):
    return series.astype(object).map(lambda v: np.nan if pd.isna(v) else float(v))


def as_text(series):
    return series.astype(object).map(lambda v: None if pd.isna(v) else str(v))


mismatches = {}
for column in [ms.TARGET] + ms.ALL_FEATURES:
    if column in ms.NUMERIC_FEATURES + ms.CLINICAL_BINARY_FEATURES:
        ours, theirs = as_float(df[column]), as_float(saved[column])
    else:
        ours, theirs = as_text(df[column]), as_text(saved[column])
    equal = (ours == theirs) | (ours.isna() & theirs.isna())
    mismatches[column] = int((~equal).sum())
print(f"{ms.DATA_PATH.name}: sha256 {cs.sha256(ms.DATA_PATH)}…")
print("Columns compared:", len(mismatches), "| cells that differ:", sum(mismatches.values()))
assert sum(mismatches.values()) == 0, {k: v for k, v in mismatches.items() if v}

cleaned_grading_dataset.xlsx: sha256 0ce9b384a51a4c47…
Columns compared: 25 | cells that differ: 0


## 6. Known limitation: "None" categories become missing when the file is loaded

Three categorical findings use the category `"None"` for "text recorded, finding absent". `pandas.read_excel` treats the string "None" as missing by default, and `ms.load_labelled_data()` (used by every model) reads the file with those defaults. So for these three features, "recorded as absent" was merged with "not recorded" in all modelling. Found during this cleanup; left unchanged so that every result stays reproducible.

In [14]:
loaded = pd.read_excel(ms.DATA_PATH)
pd.DataFrame({
    "'None' in the file": [(saved[c] == "None").sum() for c in ["Endo_Stress_Line_Location", "Stroma_Arcus_Severity", "Epi_Exposure_Severity"]],
    "'None' after loading": [(loaded[c] == "None").sum() for c in ["Endo_Stress_Line_Location", "Stroma_Arcus_Severity", "Epi_Exposure_Severity"]],
    "Paired binary feature (keeps the distinction)": ["Endo_Has_Stress_Lines", "Stroma_Has_Arcus", "Epi_Has_Exposure"],
}, index=["Endo_Stress_Line_Location", "Stroma_Arcus_Severity", "Epi_Exposure_Severity"])

,'None' in the file,'None' after loading,Paired binary feature (keeps the distinction)
Endo_Stress_Line_Location,63,0,Endo_Has_Stress_Lines
Stroma_Arcus_Severity,102,0,Stroma_Has_Arcus
Epi_Exposure_Severity,271,0,Epi_Has_Exposure


The paired binary flags still separate "absent" from "not recorded", so the information reaches the models through them. The deployed feature schema also has no "None" category, so training and serving treat these values the same way.

## 7. The labelled modelling dataset

`ms.load_labelled_data()` keeps the tissues with a reviewed grade: 1,036 (A+ 545, A 319, B 172). Missingness per feature group:

In [15]:
labelled = ms.load_labelled_data()
pd.DataFrame({
    "Features": [len(ms.NUMERIC_FEATURES), len(ms.CLINICAL_BINARY_FEATURES), len(ms.CLINICAL_CATEGORICAL_FEATURES)],
    "Missing % (range over features)": [
        f"{labelled[g].isna().mean().min() * 100:.1f}–{labelled[g].isna().mean().max() * 100:.1f}"
        for g in [ms.NUMERIC_FEATURES, ms.CLINICAL_BINARY_FEATURES, ms.CLINICAL_CATEGORICAL_FEATURES]],
}, index=["Konan morphology", "Slit-lamp binary findings", "Slit-lamp categorical findings"])

,Features,Missing % (range over features)
Konan morphology,6,9.3–9.3
Slit-lamp binary findings,13,44.7–48.9
Slit-lamp categorical findings,5,48.9–70.4


## Conclusion

- 1,036 tissues have a specialist-reviewed grade (A+ 545, A 319, B 172); 15 annotated grades were resolved by review, not by rule.
- The 24 features are 6 Konan measurements and 18 slit-lamp findings (13 binary, 5 categorical; two very sparse categorical fields were dropped in notebook 04). Impossible all-zero Konan records became missing.
- A missing finding stays missing: in the models it becomes its own category (`-1` / `__MISSING__`) for slit-lamp features and the training-fold median for Konan values (Tabular 03), never "No" or zero.
- The in-memory cleaning reproduces the saved dataset exactly.

**Limitations:** for three categorical findings, "recorded absent" (`"None"`) is read as missing at load time (Section 6); their paired binary flags keep the distinction. the free-text parsers are rule-based and were checked against the recorded phrasing, not independently validated; the SD value of one tissue has no recorded source; about 45% of tissues have no slit-lamp findings (all of them from 2023, see Tabular 02 and Hybrid 03).